# Phase 11d — Cross-Distribution Round 2: Codeforces (lowest saturation)

Tests the **saturation-magnitude corollary** added in Phase 11c: did the
+40pp pushdown gap on HumanEval+MBPP shrink to +33pp on BigCodeBench
because BCB has lower baseline saturation? If yes, Codeforces (≥2000
rating, Qwen pass-rate ~7%) should show even WEAKER pushdown and
STRONGER pushup emergence.

**Pre-registered prediction (corollary)**:
- α=−100 pushdown gap: collapses to <15pp or null
- α=+20 / α=+200 pushup gap: emerges >+20pp

**Same L31 pre_tool probe direction** — trained on Phase 6 SWE-bench Pro
N=99, top-K=10 diff-of-means.

**Compute**: 30 Codeforces prompts × 1 site × 13 αs × 2 directions = 780
forwards. ~25-30min on RTX 6000 Blackwell.

**Outcome (2026-05-09)**: corollary FALSIFIED. α=−100 pushdown +40pp,
matching HumanEval+MBPP exactly. No pushup emergence. Refined thesis:
**α=−100 robustness theorem** — saturation-INDEPENDENT.


In [ ]:
# 0) GPU pre-flight
import subprocess
out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'GPU: {out}')
mem_gb = int(out.split(',')[1].strip().split()[0]) / 1024
assert mem_gb >= 47, f'Need >=48GB. Got {mem_gb:.1f}GB.'


In [ ]:
# 1) Install
!pip install -q -U transformers
!pip install -q datasets safetensors huggingface-hub
!pip install -q flash-linear-attention causal-conv1d --no-build-isolation 2>&1 | tail -3
!pip install -q flash-attn --no-build-isolation 2>/dev/null && echo 'flash-attn ok' || echo 'flash-attn unavailable, will use sdpa'
import importlib
for pkg in ['fla', 'causal_conv1d']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg}: OK')
    except ImportError as e:
        raise SystemExit(f'BLOCKING: {pkg} missing — {e}.')


In [ ]:
# 2) Drive mount
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/openinterp_runs'
from pathlib import Path
P6 = Path(DRIVE_ROOT) / 'swebench_v6_phase6'
OUT = Path(DRIVE_ROOT) / 'phase11d_cross_distribution_round2'
OUT.mkdir(parents=True, exist_ok=True)
assert P6.exists() and (P6 / 'phase6_results.json').exists()
print(f'P6: {P6}, OUT: {OUT}')


In [ ]:
# 3) Load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = 'Qwen/Qwen3.6-27B'
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
attn_impl = 'flash_attention_2'
try:
    import flash_attn
except ImportError:
    attn_impl = 'sdpa'
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, attn_implementation=attn_impl,
    device_map={'': 0}, trust_remote_code=True,
)
model.eval()
device = next(model.parameters()).device
print(f'Loaded {sum(p.numel() for p in model.parameters())/1e9:.2f}B params on {device}')
print(f'VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')


In [ ]:
# 4) Train L31 pre_tool probe direction from Phase 6 N=99 captures
import json, warnings
warnings.filterwarnings('ignore')
import numpy as np
from safetensors.torch import load_file

CAPTURES = P6 / 'captures'

with open(P6 / 'phase6_results.json') as f:
    data = json.load(f)
results_list = data if isinstance(data, list) else (data.get('results') or list(data.values()))
labels = {}
for r in results_list:
    iid = r.get('iid') or r.get('instance_id') or r.get('id')
    patch = r.get('patch_n_bytes', 0) or r.get('patch_bytes', 0) or 0
    if iid:
        labels[iid] = int(patch > 0)
print(f'Labels: {len(labels)}, positive={sum(labels.values())}')

def load_capture(iid):
    metas = list(CAPTURES.glob(f'{iid}*.meta.json'))
    if not metas: return None
    return json.loads(metas[0].read_text()), load_file(str(metas[0].with_suffix('').with_suffix('.safetensors')))

def site_vec(m, t, layer, position):
    vecs = [t[r['activation_key']].to(torch.float32).numpy()
            for r in m['records']
            if r['layer'] == layer and r['position_label'] == position
            and r['activation_key'] in t]
    return np.mean(np.stack(vecs, axis=0), axis=0) if vecs else None

LAYER, POSITION = 31, 'pre_tool'
X, y = [], []
for iid, lab in labels.items():
    c = load_capture(iid)
    if c is None: continue
    v = site_vec(c[0], c[1], LAYER, POSITION)
    if v is not None:
        X.append(v); y.append(lab)
X, y = np.stack(X), np.array(y)
print(f'L{LAYER} {POSITION}: X shape {X.shape}, n_pos={int(y.sum())}')

diff = X[y==1].mean(axis=0) - X[y==0].mean(axis=0)
top_idx = np.argsort(-np.abs(diff))[:10]
probe_dir = np.zeros(X.shape[1], dtype=np.float32)
probe_dir[top_idx] = diff[top_idx]
probe_dir /= np.linalg.norm(probe_dir) + 1e-12

rng = np.random.default_rng(2026)
random_dir = rng.standard_normal(X.shape[1]).astype(np.float32)
random_dir /= np.linalg.norm(random_dir)

print(f'Probe direction L2={np.linalg.norm(probe_dir):.4f}')
print(f'Cosine(probe, random): {float(probe_dir @ random_dir):+.4f}')


In [ ]:
# 5) Load Codeforces rating>=2000 — lowest-saturation code regime
print('Loading open-r1/codeforces (parquet, no script)...')
from datasets import load_dataset
ds = load_dataset('open-r1/codeforces', split='train')
ds_hard = ds.filter(lambda x: (x.get('rating') or 0) >= 2000)
print(f'Codeforces rating>=2000: {len(ds_hard)} problems (full train: {len(ds)})')
ds_sample = ds_hard.shuffle(seed=2026).select(range(min(30, len(ds_hard))))

cf_prompts = []
for i, ex in enumerate(ds_sample):
    text = (ex.get('description') or ex.get('problem') or ex.get('statement') or '')[:500]
    cf_prompts.append({
        'id': f'codeforces_{i}',
        'task': text,
        'rating': ex.get('rating'),
        'source': 'codeforces',
    })
print(f'Sampled prompts: {len(cf_prompts)}')

with open(OUT / 'cf_prompts.json', 'w') as f:
    json.dump(cf_prompts, f)


In [ ]:
# 6) Build chat + steering function (mirrors v11c protocol)
import time

SYSTEM = ('You are a coding assistant. The user describes a programming task. '
          'Think step by step about how to solve it, then write the code.')

def build_chat(task_text):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Task: {task_text}\n\nProvide working code.'},
    ]
    return tok.apply_chat_template(messages, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=True)

ALPHAS = [-200.0, -100.0, -50.0, -20.0, -5.0, -2.0, 0.0,
          2.0, 5.0, 20.0, 50.0, 100.0, 200.0]
GEN_TOKENS = 40
CONTROL_TOKENS = ['def', 'the', 'we', 'I', 'a']

def steered_gen(input_ids, layer, alpha, direction_t, gen_tokens=GEN_TOKENS):
    state = {'fired': 0, 'first_logits': None}
    def hook(module, inp, out):
        is_tuple = isinstance(out, tuple)
        h = out[0] if is_tuple else out
        if state['fired'] >= 1: return out
        state['fired'] += 1
        modified = h.clone()
        if alpha != 0.0:
            modified[:, -1, :] = h[:, -1, :] + alpha * direction_t
        return (modified,) + out[1:] if is_tuple else modified
    handle = model.model.layers[layer].register_forward_hook(hook)
    try:
        with torch.no_grad():
            outputs = model(input_ids, use_cache=False)
        state['first_logits'] = outputs.logits[0, -1, :].detach().cpu().float().clone()
        state['fired'] = 0
        with torch.no_grad():
            gen_ids = model.generate(
                input_ids, max_new_tokens=gen_tokens, do_sample=False,
                pad_token_id=tok.eos_token_id, use_cache=True,
            )
        new_text = tok.decode(gen_ids[0, input_ids.shape[1]:], skip_special_tokens=False)
    finally:
        handle.remove()
    return {'new_text': new_text, 'first_logits': state['first_logits']}

control_ids = [tok.encode(t, add_special_tokens=False)[0] for t in CONTROL_TOKENS]

# Smoke test
text = build_chat(cf_prompts[0]['task'])
input_ids = tok(text, return_tensors='pt', add_special_tokens=False).input_ids.to(device)
sm = steered_gen(input_ids, LAYER, 0.0,
                 torch.from_numpy(probe_dir).to(device).to(torch.bfloat16))
print(f'Smoke L{LAYER} pre_tool α=0:')
print(f'  Generated: {sm["new_text"][:120]!r}')


In [ ]:
# 7) Full sweep on 30 Codeforces prompts × 13 αs × 2 directions
import numpy as np

def sanitize(o):
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, dict): return {k: sanitize(v) for k, v in o.items()}
    if isinstance(o, list): return [sanitize(x) for x in o]
    if isinstance(o, (np.float32, np.float64)): return float(o)
    if isinstance(o, (np.int32, np.int64)): return int(o)
    return o

probe_dir_t = torch.from_numpy(probe_dir).to(device).to(torch.bfloat16)
random_dir_t = torch.from_numpy(random_dir).to(device).to(torch.bfloat16)

results = []
t0 = time.time()
for i, pr in enumerate(cf_prompts):
    text = build_chat(pr['task'])
    input_ids = tok(text, return_tensors='pt', add_special_tokens=False).input_ids.to(device)
    base = steered_gen(input_ids, LAYER, 0.0, probe_dir_t)
    target_id = int(torch.argmax(base['first_logits']))
    qd = {'id': pr['id'], 'rating': pr.get('rating'), 'source': pr['source'],
          'baseline': base['new_text'][:200], 'target_token_id': target_id,
          'sweeps': {'probe': [], 'random': []}}
    for alpha in ALPHAS:
        for dn, dt in [('probe', probe_dir_t), ('random', random_dir_t)]:
            r = steered_gen(input_ids, LAYER, alpha, dt)
            lp = torch.log_softmax(r['first_logits'].float(), dim=-1)
            target_lp = float(lp[target_id])
            ctrl_mean = sum(float(lp[cid]) for cid in control_ids) / len(control_ids)
            qd['sweeps'][dn].append({
                'alpha': alpha,
                'new_text': r['new_text'][:200],
                'target_logprob': target_lp,
                'control_mean_logprob': ctrl_mean,
                'flipped_vs_baseline': r['new_text'].strip() != base['new_text'].strip(),
            })
    results.append(qd)
    if (i + 1) % 5 == 0:
        elapsed = (time.time() - t0) / 60
        eta = elapsed / (i + 1) * (len(cf_prompts) - i - 1)
        with open(OUT / 'partial.json', 'w') as f:
            json.dump(sanitize(results), f, indent=2)
        print(f'  [{i+1:>3d}/{len(cf_prompts)}] elapsed {elapsed:.1f}min, ETA {eta:.1f}min')

with open(OUT / 'full.json', 'w') as f:
    json.dump(sanitize(results), f, indent=2)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')


In [ ]:
# 8) Verdict — compares CF vs BCB (P11c) vs HE+MBPP (P11)
import numpy as np

def summarize_d(results, dir_name):
    rows = []
    for alpha in ALPHAS:
        flips = [int(next((s for s in r['sweeps'][dir_name] if s['alpha']==alpha), {}).get('flipped_vs_baseline', 0))
                 for r in results]
        if flips: rows.append({'alpha': alpha, 'flip_rate': float(np.mean(flips))})
    return rows
p_rows = {r['alpha']: r for r in summarize_d(results, 'probe')}
r_rows = {r['alpha']: r for r in summarize_d(results, 'random')}

# Phase 11c BCB benchmark (loaded from prior verdict if present, else hard-coded fallback)
P11C_BCB = {  # BCB gap (P11c)
    -200: 13.3, -100: 33.3, -50: 10.0, -20: -10.0, -5: 0.0, -2: -3.3,
    0: 0.0, 2: 0.0, 5: -3.3, 20: 26.7, 50: 3.3, 100: -16.7, 200: 23.3,
}
P11_HE = {  # HE+MBPP gap (P11)
    -200: 33.0, -100: 40.0, -50: 27.0, -20: 16.0, -5: 6.0, -2: 0.0,
    0: 0.0, 2: 0.0, 5: -3.0, 20: 0.0, 50: 3.0, 100: -3.0, 200: -14.0,
}

print(f'\n=== L31 pre_tool — Codeforces(P11d) vs BCB(P11c) vs HE+MBPP(P11) ===')
print(f'{"alpha":>6} {"CF_p":>7} {"CF_r":>7} {"CF_gap":>8} {"BCB_gap":>8} {"P11_gap":>8}')
verdict_d = []
for alpha in ALPHAS:
    lp = p_rows.get(alpha, {}).get('flip_rate', 0)
    lr = r_rows.get(alpha, {}).get('flip_rate', 0)
    cf_gap = (lp - lr) * 100
    bcb_gap = P11C_BCB.get(alpha, 0)
    p11_gap = P11_HE.get(alpha, 0)
    print(f'{alpha:>+6.0f} {lp*100:>7.1f} {lr*100:>7.1f} {cf_gap:>+8.1f} {bcb_gap:>+8.1f} {p11_gap:>+8.1f}')
    verdict_d.append({'alpha': alpha, 'cf_gap_pp': cf_gap,
                       'bcb_gap_pp': bcb_gap, 'p11_gap_pp': p11_gap})

push_cf = max((r['cf_gap_pp'] for r in verdict_d if r['alpha']<0), default=0)
pull_cf = max((r['cf_gap_pp'] for r in verdict_d if r['alpha']>0), default=0)
push_bcb = max((r['bcb_gap_pp'] for r in verdict_d if r['alpha']<0), default=0)
push_p11 = max((r['p11_gap_pp'] for r in verdict_d if r['alpha']<0), default=0)

if push_cf >= 25:
    msg = f'🅑 LEVER HOLDS at lowest saturation: CF pushdown +{push_cf:.0f}pp'
elif push_cf < 15 and pull_cf > 15:
    msg = f'🅑 CORROLLARY CONFIRMED: pushdown weak (+{push_cf:.0f}pp), pushup emerges (+{pull_cf:.0f}pp)'
elif push_cf >= 15:
    msg = f'🟡 ATTENUATED: CF pushdown +{push_cf:.0f}pp (lower than BCB +{push_bcb:.0f}pp)'
else:
    msg = f'⚪ NULL: CF no significant lever (push +{push_cf:.0f}pp, pull +{pull_cf:.0f}pp)'

print(f'\n{msg}')
print(f'Pushdown saturation gradient: P11 +{push_p11:.0f}pp → P11c +{push_bcb:.0f}pp → P11d +{push_cf:.0f}pp')

with open(OUT / 'verdict.json', 'w') as f:
    json.dump(sanitize({
        'site': 'L31 pre_tool',
        'dataset': 'Codeforces (rating>=2000)',
        'pushdown_max_cf': push_cf, 'pushup_max_cf': pull_cf,
        'pushdown_max_bcb': push_bcb, 'pushdown_max_p11': push_p11,
        'classification': msg, 'verdict_rows': verdict_d,
        'corollary_test': 'saturation-magnitude predicts: harder code (less saturated) → weaker pushdown OR bidirectional pattern',
        'observed_2026_05_09': 'corollary FALSIFIED: CF +40pp matches HE+MBPP, no pushup emergence; refined to α=-100 robustness theorem',
    }), f, indent=2)
print(f'\nSaved {OUT}/verdict.json')


## Interpretation map

| Outcome | Implication |
|---|---|
| 🅑 **LEVER HOLDS** (CF pushdown ≥+25pp) | Falsifies saturation-magnitude corollary. Refines paper-5 thesis to **α=-100 robustness theorem**: pushdown lever is saturation-INDEPENDENT across distributions spanning ~12× pass-rate variation. |
| 🟡 **ATTENUATED** (CF pushdown +15-25pp) | Partially supports corollary — gap shrinks at lower saturation but doesn't collapse. Mixed evidence. |
| 🅑 **CORROLLARY CONFIRMED** (push <+15pp, pull >+15pp) | Original P11c interpretation correct: lever scales with saturation, becomes bidirectional at lowest saturation. |
| ⚪ **NULL** | Probe doesn't transfer to lowest-saturation regime in either direction. L31 pre_tool is in-distribution-bound. |

## Observed (2026-05-09)

🅑 LEVER HOLDS — CF pushdown +40pp at α=-100, matching HE+MBPP exactly.
**Saturation-magnitude corollary FALSIFIED.** Refined to **α=-100 robustness theorem**.
